The `base.py` module defines the core Runnable abstraction, composition primitives, callable wrappers, bindings, mapping helpers, and conversion utilities used by LangChain Runnable pipelines.
# Classes
- `Runnable`: A unit of work that can be invoked, batched, streamed, transformed, configured, and composed with other Runnables.
- `RunnableSerializable`: A Runnable that supports serialization to JSON and runtime-configurable fields or alternatives.
- `RunnableSequence`: Runs Runnables sequentially, passing the output of each step as the input to the next step.
- `RunnableParallel`: Runs a mapping of Runnables concurrently using the same input and returns their results in a dictionary.
- `RunnableGenerator`: Wraps a synchronous or asynchronous generator function as a streaming Runnable.
- `RunnableLambda`: Converts a Python callable into a Runnable for use in LangChain sequences and pipelines.
- `RunnableEachBase`: The serializable base class that applies a bound Runnable to every item in an input sequence.
- `RunnableEach`: Applies a bound Runnable separately to each item in an input sequence.
- `RunnableBindingBase`: Delegates execution to another Runnable while merging bound arguments, configuration, and custom types.
- `RunnableBinding`: Wraps a Runnable with additional bound arguments, configuration, listeners, types, retries, or other behavior.

# Runnable

A unit of work that can be invoked, batched, streamed, transformed, configured, and composed with other Runnables.

## Attributes

1. `name`: Optional name used for debugging and tracing.
   * **Type:** `str | None`
2. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `type[Input]`
3. `OutputType`: The Python type produced as output by the Runnable.
   * **Type:** `type[Output]`
4. `input_schema`: The Pydantic model representing the Runnable input.
   * **Type:** `TypeBaseModel`
5. `output_schema`: The Pydantic model representing the Runnable output.
   * **Type:** `TypeBaseModel`
6. `config_specs`: The configurable field specifications exposed by the Runnable.
   * **Type:** `list[ConfigurableFieldSpec]`

## Methods

1. `get_name`: Returns the name of the Runnable.
   * **Syntax:**
     ```python
     get_name(
         self,
         suffix: str | None = None, # Optional suffix appended to the name
         *,
         name: str | None = None # Optional custom name
     ) -> str
     ```

2. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

3. `get_input_jsonschema`: Returns a JSON Schema representing the Runnable input.
   * **Syntax:**
     ```python
     get_input_jsonschema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> dict[str, Any]
     ```

4. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

5. `get_output_jsonschema`: Returns a JSON Schema representing the Runnable output.
   * **Syntax:**
     ```python
     get_output_jsonschema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> dict[str, Any]
     ```

6. `config_schema`: Returns a Pydantic model representing the configuration accepted by the Runnable.
   * **Syntax:**
     ```python
     config_schema(
         self,
         *,
         include: Sequence[str] | None = None # Configuration fields to include
     ) -> type[BaseModel]
     ```

7. `get_config_jsonschema`: Returns a JSON Schema representing the Runnable configuration.
   * **Syntax:**
     ```python
     get_config_jsonschema(
         self,
         *,
         include: Sequence[str] | None = None # Configuration fields to include
     ) -> dict[str, Any]
     ```

8. `get_graph`: Returns a graph representation of the Runnable.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> Graph
     ```

9. `get_prompts`: Returns the prompt templates used by the Runnable.
   * **Syntax:**
     ```python
     get_prompts(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> list[BasePromptTemplate[Any]]
     ```

10. `pipe`: Composes the current Runnable with other Runnable-like objects to create a RunnableSequence.
   * **Syntax:**
     ```python
     pipe(
         self,
         *others: Runnable[Any, Other] | Callable[[Any], Other], # Runnables or functions to compose
         name: str | None = None # Optional custom name
     ) -> RunnableSerializable[Input, Other]
     ```

11. `pick`: Selects one or more keys from the dictionary output of the Runnable.
   * **Syntax:**
     ```python
     pick(
         self,
         keys: str | list[str] # Key or list of keys to select
     ) -> RunnableSerializable[Any, Any]
     ```

12. `assign`: Adds new fields to the dictionary output of the Runnable.
   * **Syntax:**
     ```python
     assign(
         self,
         **kwargs: Runnable[dict[str, Any], Any] | Callable[[dict[str, Any]], Any] | Mapping[str, Runnable[dict[str, Any], Any] | Callable[[dict[str, Any]], Any]] # Additional arguments
     ) -> RunnableSerializable[Any, Any]
     ```

13. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Output
     ```

14. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Output
     ```

15. `batch`: Processes multiple inputs and returns their outputs as a list.
   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[Input], # Inputs passed to the Runnable
         config: RunnableConfig | list[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> list[Output]
     ```

16. `batch_as_completed`: Processes multiple inputs in parallel and yields results as they complete.
   * **Syntax:**
     ```python
     batch_as_completed(
         self,
         inputs: Sequence[Input], # Inputs passed to the Runnable
         config: RunnableConfig | Sequence[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[tuple[int, Output | Exception]]
     ```

17. `abatch`: Asynchronously processes multiple inputs and returns their outputs as a list.
   * **Syntax:**
     ```python
     abatch(
         self,
         inputs: list[Input], # Inputs passed to the Runnable
         config: RunnableConfig | list[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> list[Output]
     ```

18. `abatch_as_completed`: Asynchronously processes multiple inputs and yields results as they complete.
   * **Syntax:**
     ```python
     abatch_as_completed(
         self,
         inputs: Sequence[Input], # Inputs passed to the Runnable
         config: RunnableConfig | Sequence[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[tuple[int, Output | Exception]]
     ```

19. `stream`: Synchronously streams output from a single input.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

20. `astream`: Asynchronously streams output from a single input.
   * **Syntax:**
     ```python
     astream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

21. `astream_log`: Asynchronously streams execution logs, intermediate results, and state changes.
   * **Syntax:**
     ```python
     astream_log(
         self,
         input: Any, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         *,
         diff: bool = True, # Yield state differences instead of complete states
         with_streamed_output_list: bool = True, # Include streamed output in the log
         include_names: Sequence[str] | None = None, # Runnable names to include
         include_types: Sequence[str] | None = None, # Runnable types to include
         include_tags: Sequence[str] | None = None, # Runnable tags to include
         exclude_names: Sequence[str] | None = None, # Runnable names to exclude
         exclude_types: Sequence[str] | None = None, # Runnable types to exclude
         exclude_tags: Sequence[str] | None = None, # Runnable tags to exclude
         **kwargs: Any # Additional arguments
     ) -> AsyncIterator[RunLogPatch] | AsyncIterator[RunLog]
     ```

22. `astream_events`: Asynchronously streams lifecycle and intermediate events produced by the Runnable.
   * **Syntax:**
     ```python
     astream_events(
         self,
         input: Any, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         *,
         version: Literal['v1', 'v2', 'v3'] = 'v2', # Event schema version
         include_names: Sequence[str] | None = None, # Runnable names to include
         include_types: Sequence[str] | None = None, # Runnable types to include
         include_tags: Sequence[str] | None = None, # Runnable tags to include
         exclude_names: Sequence[str] | None = None, # Runnable names to exclude
         exclude_types: Sequence[str] | None = None, # Runnable types to exclude
         exclude_tags: Sequence[str] | None = None, # Runnable tags to exclude
         **kwargs: Any # Additional arguments
     ) -> AsyncIterator[StreamEvent] | Awaitable[Any]
     ```

23. `stream_events`: Synchronously streams lifecycle and intermediate events for supported Runnable types.
   * **Syntax:**
     ```python
     stream_events(
         self,
         input: Any, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         *,
         version: Literal['v1', 'v2', 'v3'] = 'v2', # Event schema version
         include_names: Sequence[str] | None = None, # Runnable names to include
         include_types: Sequence[str] | None = None, # Runnable types to include
         include_tags: Sequence[str] | None = None, # Runnable tags to include
         exclude_names: Sequence[str] | None = None, # Runnable names to exclude
         exclude_types: Sequence[str] | None = None, # Runnable types to exclude
         exclude_tags: Sequence[str] | None = None, # Runnable tags to exclude
         **kwargs: Any # Additional arguments
     ) -> Iterator[StreamEvent] | Iterator[Any]
     ```

24. `transform`: Synchronously transforms an iterator of inputs into an iterator of outputs.
   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

25. `atransform`: Asynchronously transforms an input stream into an output stream.
   * **Syntax:**
     ```python
     atransform(
         self,
         input: AsyncIterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

26. `bind`: Binds fixed arguments to the Runnable and returns a new Runnable.
   * **Syntax:**
     ```python
     bind(
         self,
         **kwargs: Any # Additional arguments
     ) -> Runnable[Input, Output]
     ```

27. `with_config`: Binds configuration values to the Runnable and returns a new Runnable.
   * **Syntax:**
     ```python
     with_config(
         self,
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Runnable[Input, Output]
     ```

28. `with_listeners`: Binds synchronous lifecycle listeners to the Runnable.
   * **Syntax:**
     ```python
     with_listeners(
         self,
         *,
         on_start: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None, # Listener called before execution starts
         on_end: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None, # Listener called after execution finishes
         on_error: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None # Listener called when execution fails
     ) -> Runnable[Input, Output]
     ```

29. `with_alisteners`: Binds asynchronous lifecycle listeners to the Runnable.
   * **Syntax:**
     ```python
     with_alisteners(
         self,
         *,
         on_start: AsyncListener | None = None, # Listener called before execution starts
         on_end: AsyncListener | None = None, # Listener called after execution finishes
         on_error: AsyncListener | None = None # Listener called when execution fails
     ) -> Runnable[Input, Output]
     ```

30. `with_types`: Binds custom input and output types to the Runnable.
   * **Syntax:**
     ```python
     with_types(
         self,
         *,
         input_type: type[Input] | None = None, # Custom input type
         output_type: type[Output] | None = None # Custom output type
     ) -> Runnable[Input, Output]
     ```

31. `with_retry`: Returns a Runnable that retries execution when specified exceptions occur.
   * **Syntax:**
     ```python
     with_retry(
         self,
         *,
         retry_if_exception_type: tuple[type[BaseException], ...] = (Exception,), # Exception types that trigger a retry
         wait_exponential_jitter: bool = True, # Add jitter to the retry wait time
         exponential_jitter_params: ExponentialJitterParams | None = None, # Exponential retry wait parameters
         stop_after_attempt: int = 3 # Maximum number of attempts
     ) -> Runnable[Input, Output]
     ```

32. `map`: Returns a Runnable that applies the current Runnable to every item in a sequence.
   * **Syntax:**
     ```python
     map(
         self
     ) -> Runnable[Sequence[Input], list[Output]]
     ```

33. `with_fallbacks`: Adds fallback Runnables that are tried when the original Runnable fails.
   * **Syntax:**
     ```python
     with_fallbacks(
         self,
         fallbacks: Sequence[Runnable[Input, Output]], # Fallback Runnables to try
         *,
         exceptions_to_handle: tuple[type[BaseException], ...] = (Exception,), # Exception types handled by the fallbacks
         exception_key: str | None = None # Input key used to pass the exception
     ) -> RunnableWithFallbacksT[Input, Output]
     ```

34. `as_tool`: Converts the Runnable into a LangChain BaseTool.
   * **Syntax:**
     ```python
     as_tool(
         self,
         args_schema: type[BaseModel] | None = None, # Pydantic schema for the generated tool
         *,
         name: str | None = None, # Optional custom name
         description: str | None = None, # Description of the generated tool
         arg_types: dict[str, type] | None = None # Tool argument names and types
     ) -> BaseTool
     ```

# RunnableSerializable

A Runnable that supports serialization to JSON and runtime-configurable fields or alternatives.

### Attributes

1. `name`: Optional name used for debugging and tracing.
   * **Type:** `str | None`

### Methods

1. `to_json`: Serializes the Runnable to a JSON-compatible LangChain representation.
   * **Syntax:**
     ```python
     to_json(
         self
     ) -> SerializedConstructor | SerializedNotImplemented
     ```

2. `configurable_fields`: Marks selected model fields as configurable at runtime.
   * **Syntax:**
     ```python
     configurable_fields(
         self,
         **kwargs: AnyConfigurableField # Additional arguments
     ) -> RunnableSerializable[Input, Output]
     ```

3. `configurable_alternatives`: Defines alternative Runnables that can be selected through runtime configuration.
   * **Syntax:**
     ```python
     configurable_alternatives(
         self,
         which: ConfigurableField, # Configurable field used to select an alternative
         *,
         default_key: str = 'default', # Key of the default alternative
         prefix_keys: bool = False, # Prefix alternative keys with the configurable field ID
         **kwargs: Runnable[Input, Output] | Callable[[], Runnable[Input, Output]] # Additional arguments
     ) -> RunnableSerializable[Input, Output]
     ```

# RunnableSequence

Runs Runnables sequentially, passing the output of each step as the input to the next step.

### Attributes

1. `first`: The first Runnable in a sequence.
   * **Type:** `Runnable[Input, Any]`
2. `middle`: The intermediate Runnables in a sequence.
   * **Type:** `list[Runnable[Any, Any]]`
3. `last`: The final Runnable in a sequence.
   * **Type:** `Runnable[Any, Output]`
4. `steps`: All Runnables in the sequence in execution order.
   * **Type:** `list[Runnable[Any, Any]]`
5. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `type[Input]`
6. `OutputType`: The Python type produced as output by the Runnable.
   * **Type:** `type[Output]`
7. `config_specs`: The configurable field specifications exposed by the Runnable.
   * **Type:** `list[ConfigurableFieldSpec]`

### Methods

1. `__init__`: Initializes the class instance.
   * **Syntax:**
     ```python
     __init__(
         self,
         *steps: RunnableLike[Any, Any], # Runnables that make up the sequence
         name: str | None = None, # Optional custom name
         first: Runnable[Any, Any] | None = None, # First Runnable in the sequence
         middle: list[Runnable[Any, Any]] | None = None, # Middle Runnables in the sequence
         last: Runnable[Any, Any] | None = None # Last Runnable in the sequence
     ) -> None
     ```

2. `get_lc_namespace`: Returns the LangChain serialization namespace for the class.
   * **Syntax:**
     ```python
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

3. `is_lc_serializable`: Indicates whether the class supports LangChain serialization.
   * **Syntax:**
     ```python
     is_lc_serializable(
         cls
     ) -> bool
     ```

4. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

5. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

6. `get_graph`: Returns a graph representation of the Runnable.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> Graph
     ```

7. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Output
     ```

8. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Output
     ```

9. `batch`: Processes multiple inputs and returns their outputs as a list.
   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[Input], # Inputs passed to the Runnable
         config: RunnableConfig | list[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> list[Output]
     ```

10. `abatch`: Asynchronously processes multiple inputs and returns their outputs as a list.
   * **Syntax:**
     ```python
     abatch(
         self,
         inputs: list[Input], # Inputs passed to the Runnable
         config: RunnableConfig | list[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> list[Output]
     ```

11. `transform`: Synchronously transforms an iterator of inputs into an iterator of outputs.
   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

12. `stream`: Synchronously streams output from a single input.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

13. `atransform`: Asynchronously transforms an input stream into an output stream.
   * **Syntax:**
     ```python
     atransform(
         self,
         input: AsyncIterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

14. `astream`: Asynchronously streams output from a single input.
   * **Syntax:**
     ```python
     astream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

# RunnableParallel

Runs a mapping of Runnables concurrently using the same input and returns their results in a dictionary.

### Attributes

1. `steps__`: A mapping of output keys to Runnables executed in parallel.
   * **Type:** `Mapping[str, Runnable[Input, Any]]`
2. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `Any`
3. `config_specs`: The configurable field specifications exposed by the Runnable.
   * **Type:** `list[ConfigurableFieldSpec]`

### Methods

1. `__init__`: Initializes the class instance.
   * **Syntax:**
     ```python
     __init__(
         self,
         steps__: Mapping[str, Runnable[Input, Any] | Callable[[Input], Any] | Mapping[str, Runnable[Input, Any] | Callable[[Input], Any]]] | None = None, # Mapping of output keys to Runnables
         **kwargs: Runnable[Input, Any] | Callable[[Input], Any] | Mapping[str, Runnable[Input, Any] | Callable[[Input], Any]] # Additional arguments
     ) -> None
     ```

2. `is_lc_serializable`: Indicates whether the class supports LangChain serialization.
   * **Syntax:**
     ```python
     is_lc_serializable(
         cls
     ) -> bool
     ```

3. `get_lc_namespace`: Returns the LangChain serialization namespace for the class.
   * **Syntax:**
     ```python
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

4. `get_name`: Returns the name of the Runnable.
   * **Syntax:**
     ```python
     get_name(
         self,
         suffix: str | None = None, # Optional suffix appended to the name
         *,
         name: str | None = None # Optional custom name
     ) -> str
     ```

5. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

6. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> type[BaseModel]
     ```

7. `get_graph`: Returns a graph representation of the Runnable.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> Graph
     ```

8. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> dict[str, Any]
     ```

9. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> dict[str, Any]
     ```

10. `transform`: Synchronously transforms an iterator of inputs into an iterator of outputs.
   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Iterator[dict[str, Any]]
     ```

11. `stream`: Synchronously streams output from a single input.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[dict[str, Any]]
     ```

12. `atransform`: Asynchronously transforms an input stream into an output stream.
   * **Syntax:**
     ```python
     atransform(
         self,
         input: AsyncIterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> AsyncIterator[dict[str, Any]]
     ```

13. `astream`: Asynchronously streams output from a single input.
   * **Syntax:**
     ```python
     astream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[dict[str, Any]]
     ```

# RunnableGenerator

Wraps a synchronous or asynchronous generator function as a streaming Runnable.

### Attributes

1. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `Any`
2. `OutputType`: The Python type produced as output by the Runnable.
   * **Type:** `Any`

### Methods

1. `__init__`: Initializes the class instance.
   * **Syntax:**
     ```python
     __init__(
         self,
         transform: Callable[[Iterator[Input]], Iterator[Output]] | Callable[[AsyncIterator[Input]], AsyncIterator[Output]], # Generator function used for synchronous or asynchronous transformation
         atransform: Callable[[AsyncIterator[Input]], AsyncIterator[Output]] | None = None, # Optional asynchronous generator function
         *,
         name: str | None = None # Optional custom name
     ) -> None
     ```

2. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> type[BaseModel]
     ```

3. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> type[BaseModel]
     ```

4. `transform`: Synchronously transforms an iterator of inputs into an iterator of outputs.
   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Iterator[Output]
     ```

5. `stream`: Synchronously streams output from a single input.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Iterator[Output]
     ```

6. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Output
     ```

7. `atransform`: Asynchronously transforms an input stream into an output stream.
   * **Syntax:**
     ```python
     atransform(
         self,
         input: AsyncIterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> AsyncIterator[Output]
     ```

8. `astream`: Asynchronously streams output from a single input.
   * **Syntax:**
     ```python
     astream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> AsyncIterator[Output]
     ```

9. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Output
     ```

# RunnableLambda

Converts a Python callable into a Runnable for use in LangChain sequences and pipelines.

### Attributes

1. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `Any`
2. `OutputType`: The Python type produced as output by the Runnable.
   * **Type:** `Any`
3. `config_specs`: The configurable field specifications exposed by the Runnable.
   * **Type:** `list[ConfigurableFieldSpec]`

### Methods

1. `__init__`: Initializes the class instance.
   * **Syntax:**
     ```python
     __init__(
         self,
         func: Callable[[Input], Iterator[Output]] | Callable[[Input], Runnable[Input, Output]] | Callable[[Input], Output] | Callable[[Input, RunnableConfig], Output] | Callable[[Input, CallbackManagerForChainRun], Output] | Callable[[Input, CallbackManagerForChainRun, RunnableConfig], Output] | Callable[[Input], Awaitable[Output]] | Callable[[Input], AsyncIterator[Output]] | Callable[[Input, RunnableConfig], Awaitable[Output]] | Callable[[Input, AsyncCallbackManagerForChainRun], Awaitable[Output]] | Callable[[Input, AsyncCallbackManagerForChainRun, RunnableConfig], Awaitable[Output]], # Callable to wrap as a Runnable
         afunc: Callable[[Input], Awaitable[Output]] | Callable[[Input], AsyncIterator[Output]] | Callable[[Input, RunnableConfig], Awaitable[Output]] | Callable[[Input, AsyncCallbackManagerForChainRun], Awaitable[Output]] | Callable[[Input, AsyncCallbackManagerForChainRun, RunnableConfig], Awaitable[Output]] | None = None, # Optional asynchronous callable
         name: str | None = None # Optional custom name
     ) -> None
     ```

2. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

3. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> type[BaseModel]
     ```

4. `deps`: The dependencies of this `Runnable`.
   * **Syntax:**
     ```python
     deps(
         self
     ) -> list[Runnable[Any, Any]]
     ```

5. `get_graph`: Returns a graph representation of the Runnable.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> Graph
     ```

6. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Output
     ```

7. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Output
     ```

8. `transform`: Synchronously transforms an iterator of inputs into an iterator of outputs.
   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

9. `stream`: Synchronously streams output from a single input.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

10. `atransform`: Asynchronously transforms an input stream into an output stream.
   * **Syntax:**
     ```python
     atransform(
         self,
         input: AsyncIterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

11. `astream`: Asynchronously streams output from a single input.
   * **Syntax:**
     ```python
     astream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

# RunnableEachBase

The serializable base class that applies a bound Runnable to every item in an input sequence.

### Attributes

1. `bound`: The wrapped Runnable.
   * **Type:** `Runnable[Input, Output]`
2. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `Any`
3. `OutputType`: The Python type produced as output by the Runnable.
   * **Type:** `type[list[Output]]`
4. `config_specs`: The configurable field specifications exposed by the Runnable.
   * **Type:** `list[ConfigurableFieldSpec]`

### Methods

1. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> type[BaseModel]
     ```

2. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> type[BaseModel]
     ```

3. `get_graph`: Returns a graph representation of the Runnable.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> Graph
     ```

4. `is_lc_serializable`: Indicates whether the class supports LangChain serialization.
   * **Syntax:**
     ```python
     is_lc_serializable(
         cls
     ) -> bool
     ```

5. `get_lc_namespace`: Returns the LangChain serialization namespace for the class.
   * **Syntax:**
     ```python
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

6. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Sequence[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> list[Output]
     ```

7. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Sequence[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> list[Output]
     ```

8. `astream_events`: Asynchronously streams lifecycle and intermediate events produced by the Runnable.
   * **Syntax:**
     ```python
     astream_events(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         *,
         version: Literal['v1', 'v2', 'v3'] = 'v2', # Event schema version
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[StreamEvent] | Awaitable[Any]
     ```


# RunnableEach

Applies a bound Runnable separately to each item in an input sequence.

### Methods

1. `get_name`: Returns the name of the Runnable.
   * **Syntax:**
     ```python
     get_name(
         self,
         suffix: str | None = None, # Optional suffix appended to the name
         *,
         name: str | None = None # Optional custom name
     ) -> str
     ```

2. `bind`: Binds fixed arguments to the Runnable and returns a new Runnable.
   * **Syntax:**
     ```python
     bind(
         self,
         **kwargs: Any # Additional arguments
     ) -> RunnableEach[Input, Output]
     ```

3. `with_config`: Binds configuration values to the Runnable and returns a new Runnable.
   * **Syntax:**
     ```python
     with_config(
         self,
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> RunnableEach[Input, Output]
     ```

4. `with_listeners`: Binds synchronous lifecycle listeners to the Runnable.
   * **Syntax:**
     ```python
     with_listeners(
         self,
         *,
         on_start: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None, # Listener called before execution starts
         on_end: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None, # Listener called after execution finishes
         on_error: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None # Listener called when execution fails
     ) -> RunnableEach[Input, Output]
     ```

5. `with_alisteners`: Binds asynchronous lifecycle listeners to the Runnable.
   * **Syntax:**
     ```python
     with_alisteners(
         self,
         *,
         on_start: AsyncListener | None = None, # Listener called before execution starts
         on_end: AsyncListener | None = None, # Listener called after execution finishes
         on_error: AsyncListener | None = None # Listener called when execution fails
     ) -> RunnableEach[Input, Output]
     ```

# RunnableBindingBase

Delegates execution to another Runnable while merging bound arguments, configuration, and custom types.

### Attributes

1. `bound`: The wrapped Runnable.
   * **Type:** `Runnable[Input, Output]`
2. `kwargs`: Arguments permanently bound to the wrapped Runnable.
   * **Type:** `Mapping[str, Any]`
3. `config`: Configuration permanently bound to the wrapped Runnable.
   * **Type:** `RunnableConfig`
4. `config_factories`: Functions that create additional runtime configuration.
   * **Type:** `list[Callable[[RunnableConfig], RunnableConfig]]`
5. `custom_input_type`: An optional custom input type for the binding.
   * **Type:** `Any | None`
6. `custom_output_type`: An optional custom output type for the binding.
   * **Type:** `Any | None`
7. `InputType`: The Python type accepted as input by the Runnable.
   * **Type:** `type[Input]`
8. `OutputType`: The Python type produced as output by the Runnable.
   * **Type:** `type[Output]`
9. `config_specs`: The configurable field specifications exposed by the Runnable.
   * **Type:** `list[ConfigurableFieldSpec]`

### Methods

1. `__init__`: Initializes the class instance.
   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         bound: Runnable[Input, Output], # Runnable being wrapped
         kwargs: Mapping[str, Any] | None = None, # Additional arguments
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         config_factories: list[Callable[[RunnableConfig], RunnableConfig]] | None = None, # Functions that generate runtime configurations
         custom_input_type: type[Input] | BaseModel | None = None, # Optional custom input type
         custom_output_type: type[Output] | BaseModel | None = None, # Optional custom output type
         **other_kwargs: Any # Additional model fields
     ) -> None
     ```

2. `get_name`: Returns the name of the Runnable.
   * **Syntax:**
     ```python
     get_name(
         self,
         suffix: str | None = None, # Optional suffix appended to the name
         *,
         name: str | None = None # Optional custom name
     ) -> str
     ```

3. `get_input_schema`: Returns a Pydantic model used to validate the Runnable input.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

4. `get_output_schema`: Returns a Pydantic model used to validate the Runnable output.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> TypeBaseModel
     ```

5. `get_graph`: Returns a graph representation of the Runnable.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used by the Runnable
     ) -> Graph
     ```

6. `is_lc_serializable`: Indicates whether the class supports LangChain serialization.
   * **Syntax:**
     ```python
     is_lc_serializable(
         cls
     ) -> bool
     ```

7. `get_lc_namespace`: Returns the LangChain serialization namespace for the class.
   * **Syntax:**
     ```python
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

8. `invoke`: Synchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Output
     ```

9. `ainvoke`: Asynchronously transforms a single input into an output.
   * **Syntax:**
     ```python
     ainvoke(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Output
     ```

10. `batch`: Processes multiple inputs and returns their outputs as a list.
   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[Input], # Inputs passed to the Runnable
         config: RunnableConfig | list[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> list[Output]
     ```

11. `abatch`: Asynchronously processes multiple inputs and returns their outputs as a list.
   * **Syntax:**
     ```python
     abatch(
         self,
         inputs: list[Input], # Inputs passed to the Runnable
         config: RunnableConfig | list[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> list[Output]
     ```

12. `batch_as_completed`: Processes multiple inputs in parallel and yields results as they complete.
   * **Syntax:**
     ```python
     batch_as_completed(
         self,
         inputs: Sequence[Input], # Inputs passed to the Runnable
         config: RunnableConfig | Sequence[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[tuple[int, Output | Exception]]
     ```

13. `abatch_as_completed`: Asynchronously processes multiple inputs and yields results as they complete.
   * **Syntax:**
     ```python
     abatch_as_completed(
         self,
         inputs: Sequence[Input], # Inputs passed to the Runnable
         config: RunnableConfig | Sequence[RunnableConfig] | None = None, # Configuration used by the Runnable
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[tuple[int, Output | Exception]]
     ```

14. `stream`: Synchronously streams output from a single input.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> Iterator[Output]
     ```

15. `astream`: Asynchronously streams output from a single input.
   * **Syntax:**
     ```python
     astream(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[Output]
     ```

16. `stream_events`: Synchronously streams lifecycle and intermediate events for supported Runnable types.
   * **Syntax:**
     ```python
     stream_events(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         *,
         version: Literal['v1', 'v2', 'v3'] = 'v2', # Event schema version
         **kwargs: Any # Additional arguments
     ) -> Iterator[StreamEvent] | Any
     ```

17. `astream_events`: Asynchronously streams lifecycle and intermediate events produced by the Runnable.
   * **Syntax:**
     ```python
     astream_events(
         self,
         input: Input, # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any | None # Additional arguments
     ) -> AsyncIterator[StreamEvent] | Awaitable[Any]
     ```

18. `transform`: Synchronously transforms an iterator of inputs into an iterator of outputs.
   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Iterator[Output]
     ```

19. `atransform`: Asynchronously transforms an input stream into an output stream.
   * **Syntax:**
     ```python
     atransform(
         self,
         input: AsyncIterator[Input], # Input passed to the Runnable
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> AsyncIterator[Output]
     ```


# RunnableBinding

Wraps a Runnable with additional bound arguments, configuration, listeners, types, retries, or other behavior.

### Methods

1. `bind`: Binds fixed arguments to the Runnable and returns a new Runnable.
   * **Syntax:**
     ```python
     bind(
         self,
         **kwargs: Any # Additional arguments
     ) -> Runnable[Input, Output]
     ```

2. `with_config`: Binds configuration values to the Runnable and returns a new Runnable.
   * **Syntax:**
     ```python
     with_config(
         self,
         config: RunnableConfig | None = None, # Configuration used by the Runnable
         **kwargs: Any # Additional arguments
     ) -> Runnable[Input, Output]
     ```

3. `with_listeners`: Binds synchronous lifecycle listeners to the Runnable.
   * **Syntax:**
     ```python
     with_listeners(
         self,
         *,
         on_start: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None, # Listener called before execution starts
         on_end: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None, # Listener called after execution finishes
         on_error: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None # Listener called when execution fails
     ) -> Runnable[Input, Output]
     ```

4. `with_types`: Binds custom input and output types to the Runnable.
   * **Syntax:**
     ```python
     with_types(
         self,
         input_type: type[Input] | BaseModel | None = None, # Custom input type
         output_type: type[Output] | BaseModel | None = None # Custom output type
     ) -> Runnable[Input, Output]
     ```

5. `with_retry`: Returns a Runnable that retries execution when specified exceptions occur.
   * **Syntax:**
     ```python
     with_retry(
         self,
         **kwargs: Any # Additional arguments
     ) -> Runnable[Input, Output]
     ```
